# Dataset Generator (Knowledge Distillation)

This script uses our massive "Teacher" model (`gpt-oss-120b`) to read your legal database and automatically write hundreds of high-quality Question & Answer pairs.

It then perfectly splits the data into **Train (80%)**, **Validation (10%)**, and **Test (10%)** datasets so you can fine-tune your offline `LLM'` on Google Colab.

In [1]:
!pip install -q groq chromadb tqdm rich

In [2]:
import os
import json
import random
import time
from pathlib import Path
import chromadb
from groq import Groq
from tqdm.notebook import tqdm
from rich import print

# ═══════════════════════════════════════════════
#  PASTE YOUR GROQ API KEY HERE
# ═══════════════════════════════════════════════
os.environ["GROQ_API_KEY"] = "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"

PROJECT_ROOT = Path("/Users/ritesh/Downloads/IP_Shakti_Sahayak")
CHROMA_DB_PATH = PROJECT_ROOT / "chromadb_store"
COLLECTION_NAME = "ip_sakti_legal_corpus"

LLM_MODEL = "openai/gpt-oss-120b"  # The Teacher Model

# ⚙️ GENERATION SETTINGS
NUM_CHUNKS_TO_PROCESS = 300

print("✅ Configuration loaded.")

✅ Configuration loaded.

## Load ChromaDB and Fetch Raw Legal Text

In [4]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))
collection = chroma_client.get_collection(name=COLLECTION_NAME)

# Get the total count without downloading the data
total_chunks = collection.count()
print(f"📚 Found {total_chunks} total legal chunks in ChromaDB.")

selected_docs = []
selected_metas = []

# Fetch randomly selected chunks one at a time to prevent database crashes
print(f"🎯 Fetching {NUM_CHUNKS_TO_PROCESS} random chunks...")
for _ in range(NUM_CHUNKS_TO_PROCESS):
    random_offset = random.randint(0, total_chunks - 1)
    
    # Fetch exactly 1 document at the random offset
    result = collection.get(
        limit=1, 
        offset=random_offset, 
        include=["documents", "metadatas"]
    )
    
    selected_docs.append(result["documents"][0])
    selected_metas.append(result["metadatas"][0])

print(f"✅ Successfully loaded {len(selected_docs)} random chunks for dataset generation.")

📚 Found 49689 total legal chunks in ChromaDB.

🎯 Fetching 300 random chunks...

✅ Successfully loaded 300 random chunks for dataset generation.

## The Generator Function (Teacher LLM)

In [5]:
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def generate_qa_from_text(context_text: str, metadata: dict):
    """Asks the Teacher LLM to write training Q&A pairs based on the text."""
    
    sys_prompt = """You are an expert AI creating a training dataset for a legal AI assistant.
Your job is to read the provided legal text and generate exactly 2 highly realistic Question & Answer pairs.
The questions should sound like they come from researchers, patent applicants, or lawyers.
The answers must be detailed, cite the text, and be highly accurate.

OUTPUT FORMAT: You MUST return ONLY a valid JSON array of objects, with keys 'question' and 'answer'.
Example:
[
  {"question": "Can I patent a traditional herb?", "answer": "Under Section 3(p)..."},
  {"question": "What form do I need for AYUSH manufacturing?", "answer": "You must file Form 25D..."}
]"""

    user_prompt = f"Source Document: {metadata.get('source_file', 'Unknown')}\n\nLegal Text:\n{context_text}\n\nGenerate 2 Q&A pairs in JSON array format."

    try:
        response = groq_client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"}, # Forces valid JSON
            temperature=0.3
        )
        
        content = response.choices[0].message.content
        
        # Handle JSON object wrapper if the LLM adds it
        parsed = json.loads(content)
        if isinstance(parsed, dict):
            # Sometimes models return {"qa_pairs": [...]} instead of just the array
            for key in parsed.keys():
                if isinstance(parsed[key], list):
                    return parsed[key]
            return []
        return parsed
        
    except Exception as e:
        print(f"⚠️ Error generating QA: {e}")
        return []

## Run the Generation Loop
 it calls the Groq API for each chunk.

In [6]:
raw_dataset = []

print("🚀 Starting Q&A Generation...")
for doc, meta in tqdm(zip(selected_docs, selected_metas), total=len(selected_docs)):
    qa_pairs = generate_qa_from_text(doc, meta)
    
    for qa in qa_pairs:
        # Format in the standard 'ChatML / ShareGPT' format required for Fine-Tuning
        formatted_example = {
            "messages": [
                {"role": "system", "content": "You are IP-SAKTI, an expert legal AI for Indian IP and Ayurvedic regulation."},
                {"role": "user", "content": qa.get("question", "")},
                {"role": "assistant", "content": qa.get("answer", "")}
            ]
        }
        raw_dataset.append(formatted_example)
    
    # Small pause to avoid hitting API rate limits
    time.sleep(1)

print(f"✅ Successfully generated {len(raw_dataset)} training examples!")

🚀 Starting Q&A Generation...

  0%|          | 0/300 [00:00<?, ?it/s]

⚠️ Error generating QA: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b`
in organization `org_01m1nnp4pne2f8gk7dhw159kv8` service tier `on_demand` on tokens per day (TPD): Limit 200000, 
Used 198932, Requested 1255. Please try again in 1m20.783999999s. Need more tokens? Upgrade to Dev Tier today at 
https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

⚠️ Error generating QA: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b`
in organization `org_01m1nnp4pne2f8gk7dhw159kv8` service tier `on_demand` on tokens per day (TPD): Limit 200000, 
Used 198929, Requested 1239. Please try again in 1m12.576s. Need more tokens? Upgrade to Dev Tier today at 
https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

⚠️ Error generating QA: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b`
in organization `org_01m1nnp4pne2f8gk7dhw159kv8` service tier `on_demand` on tokens per day (TPD): Limit 200000, 
Used 198927, Requested 1412. Please try again in 2m26.447999999s. Need more tokens? Upgrade to Dev Tier today at 
https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

...

⚠️ Error generating QA: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b`
in organization `org_01m1nnp4pne2f8gk7dhw159kv8` service tier `on_demand` on tokens per day (TPD): Limit 200000, 
Used 199726, Requested 1255. Please try again in 7m3.792s. Need more tokens? Upgrade to Dev Tier today at 
https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

⚠️ Error generating QA: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b`
in organization `org_01m1nnp4pne2f8gk7dhw159kv8` service tier `on_demand` on tokens per day (TPD): Limit 200000, 
Used 199724, Requested 1394. Please try again in 8m2.976s. Need more tokens? Upgrade to Dev Tier today at 
https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

⚠️ Error generating QA: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b`
in organization `org_01m1nnp4pne2f8gk7dhw159kv8` service tier `on_demand` on tokens per day (TPD): Limit 200000, 
Used 199721, Requested 1360. Please try again in 7m46.992s. Need more tokens? Upgrade to Dev Tier today at 
https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

⚠️ Error generating QA: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b`
in organization `org_01m1nnp4pne2f8gk7dhw159kv8` service tier `on_demand` on tokens per day (TPD): Limit 200000, 
Used 199719, Requested 1364. Please try again in 7m47.856s. Need more tokens? Upgrade to Dev Tier today at 
https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

✅ Successfully generated 442 training examples!

## Train / Validation / Test Split (80-10-10)
maximize the Train dataset, while keeping validation and test sets for evaluation.

In [7]:
# Shuffle the dataset to ensure a random mix of topics in all splits
random.shuffle(raw_dataset)

total = len(raw_dataset)
train_idx = int(total * 0.80)  # 80% for Training
val_idx = int(total * 0.90)    # 10% for Validation

train_data = raw_dataset[:train_idx]
val_data = raw_dataset[train_idx:val_idx]
test_data = raw_dataset[val_idx:]

print("📊 DATASET SPLIT RESULTS:")
print(f"  ➔ Train Data      : {len(train_data)} examples (80%)")
print(f"  ➔ Validation Data : {len(val_data)} examples (10%)")
print(f"  ➔ Test Data       : {len(test_data)} examples (10%)")

📊 DATASET SPLIT RESULTS:

➔ Train Data      : 353 examples (80%)

➔ Validation Data : 44 examples (10%)

➔ Test Data       : 45 examples (10%)

## Save Datasets to Disk (.jsonl format)

In [8]:
dataset_dir = PROJECT_ROOT / "finetuning_dataset"
dataset_dir.mkdir(exist_ok=True)

def save_jsonl(data: list, filename: str):
    filepath = dataset_dir / filename
    with open(filepath, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"💾 Saved: {filepath}")

save_jsonl(train_data, "train.jsonl")
save_jsonl(val_data, "validation.jsonl")
save_jsonl(test_data, "test.jsonl")

print("\n🎉 ALL DONE! You can now upload these 3 files to Google Colab for fine-tuning.")

💾 Saved: /Users/ritesh/Downloads/IP_Shakti_Sahayak/finetuning_dataset/train.jsonl

💾 Saved: /Users/ritesh/Downloads/IP_Shakti_Sahayak/finetuning_dataset/validation.jsonl

💾 Saved: /Users/ritesh/Downloads/IP_Shakti_Sahayak/finetuning_dataset/test.jsonl

🎉 ALL DONE! You can now upload these 3 files to Google Colab for fine-tuning.